In [1]:
!pip install -q bm25s ranx sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.5/75.5 kB 727.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.3/99.3 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 462.5/462.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.8/947.8 kB 47.4 MB/s eta 0:00:00


In [3]:
import os

PROJECT_DIR = '/content/drive/MyDrive/Colab Notebooks/Rag project'

try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.chdir(PROJECT_DIR)
except ImportError:
    print('Not running in Colab: using the current directory')

print('Working directory:', os.getcwd())

Mounted at /content/drive
Working directory: /content/drive/MyDrive/Colab Notebooks/Rag project


In [6]:
import json
import os
import yaml
import bm25s
import numpy as np
import pandas as pd
import torch
from ranx import Qrels, Run, compare, evaluate, fuse
from pathlib import Path
from sentence_transformers import SentenceTransformer
from transformers import AutoModel, AutoTokenizer

import utils


DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

Device: cuda


In [7]:
CORPUS_FILE = 'corpus_2026-10-09.jsonl'
QUESTIONS_FILE = 'vlm_eval_questions_combined.xlsx'
JUDGMENTS_FILE = 'pool_judgments.csv'   # created by you after judging the pool
INDEX_DIR = 'index'
EMB_DIR = 'embeddings'
OUT_DIR = 'retrieval_eval'

REBUILD_BM25 = True          # rebuild BM25 on title + abstract
BM25_STOPWORDS = 'en'        # must be identical at indexing and query time
DENSE_MODELS = ['medcpt', 'bge', 'qwen3']

K_RETRIEVE = 30              # candidates per retriever (the reranker will see these later)
K_POOL = 10                  # top-k from each run that enters the judging pool
METRICS = ['recall@5', 'recall@10', 'ndcg@10', 'mrr@10', 'precision@5']

os.makedirs(OUT_DIR, exist_ok=True)

In [8]:
_cfg = yaml.safe_load(Path("config.yml").read_text())
ST_CONFIGS = _cfg["ST_CONFIGS"]
MEDCPT = _cfg["MEDCPT"]

# Inject configurations and device into the imported utils module
utils.ST_CONFIGS = ST_CONFIGS
utils.MEDCPT = MEDCPT
utils.DEVICE = DEVICE

In [11]:
records = utils.load_corpus(CORPUS_FILE)

# Safely handle if records is already returned as a dictionary, or as a list
if isinstance(records, dict):
    corpus = records
else:
    corpus = {r['pmid']: r for r in records}

titles = {p: r.get('title') or '' for p, r in corpus.items()}
print(f'{len(corpus)} documents')

100 documents


In [12]:
# load the questions
qdf = pd.read_excel(QUESTIONS_FILE, dtype=str).fillna('')
if 'row_type' in qdf.columns:
    qdf = qdf[qdf.row_type == 'question'] # there is false positive and questions -> filtering to questions
qdf = qdf.reset_index(drop=True)

split = lambda s: [p for p in s.split(';') if p]
qdf['seeds'] = qdf.seed_pmids.map(split)
qdf['traps'] = qdf.fp_target_pmids.map(split)

queries = dict(zip(qdf.qid, qdf.question))
seeds = dict(zip(qdf.qid, qdf.seeds))
traps = dict(zip(qdf.qid, qdf.traps))
qtype = dict(zip(qdf.qid, qdf.type))

print(f'{len(queries)} questions')
print(qdf.type.value_counts())


48 questions
type
single_specific           15
multi_paper               15
single_paraphrased        10
hard_negative_contrast     4
unanswerable               4
Name: count, dtype: int64


In [13]:
# Every seed and trap PMID should exist in the corpus
missing = [(q, p) for q in queries for p in seeds[q] + traps[q] if p not in corpus]
print('Missing PMIDs:', missing if missing else 'none')

Missing PMIDs: none


In [14]:
# loading the bm25 frequencies
bm25 = bm25s.BM25.load(INDEX_DIR)
pmids_path = os.path.join(INDEX_DIR, 'pmids.json')
if os.path.exists(pmids_path):
   bm25_pmids = json.load(open(pmids_path))
else:  # file written by the original notebook
   bm25_pmids = [l.strip() for l in open(os.path.join(INDEX_DIR, 'docs_ind.txt')) if l.strip()]
print(f'Loaded BM25 index: {len(bm25_pmids)} docs')

Loaded BM25 index: 100 docs


In [15]:
def bm25_search(query_list, k=K_RETRIEVE):
    """One ranked list of (pmid, score) per query."""
    k = min(k, len(bm25_pmids))
    q_tokens = bm25s.tokenize(query_list, stopwords=BM25_STOPWORDS, return_ids=False)
    idx, scores = bm25.retrieve(q_tokens, k=k, show_progress=False)
    return [[(bm25_pmids[j], float(s)) for j, s in zip(idx[i], scores[i])]
            for i in range(len(query_list))]


# Quick test
for rank, (p, s) in enumerate(bm25_search(['GPT-4o on fundus photographs'], k=5)[0], 1):
    print(f'{rank}. [{p}] {s:.2f}  {titles[p][:90]}')

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

Reconstructing token strings:   0%|          | 0/1 [00:00<?, ?it/s]

1. [42640646] 4.07  Diagnostic Accuracy of Multimodal Large Language Models in Retinal Fundus Photography.
2. [42718906] 2.98  Large Language Models Approximate Inter-Expert Agreement in Glaucoma Suspect and Glaucoma 
3. [42684491] 2.48  Evaluation of a cornea-specialized large language model for diagnostic and management accu
4. [42647860] 2.27  Large Language Models for Patient Education in Cardiovascular Imaging: Prospective Observa
5. [42593567] 2.22  Large language models for lumbar spondylolisthesis detection: a multi-center pilot compara


In [16]:
emb_pmids = json.load(open(os.path.join(EMB_DIR, 'pmids.json')))

# BM25 and embeddings must come from the same corpus
assert set(emb_pmids) == set(bm25_pmids), 'BM25 index and embeddings cover different documents'
print(f'Embeddings cover {len(emb_pmids)} docs; aligned with BM25')


def dense_search(key, query_list, k=K_RETRIEVE, encoder=None):
    """One ranked list of (pmid, cosine score) per query."""
    emb = np.load(os.path.join(EMB_DIR, f'emb_{key}.npy'))
    assert emb.shape[0] == len(emb_pmids), f'{key}: rows do not match pmids.json'
    encoder = encoder or utils.QueryEncoder(key)
    S = encoder.encode(query_list) @ emb.T
    out = []
    for i in range(len(query_list)):
        top = np.argsort(-S[i])[:k]
        out.append([(emb_pmids[j], float(S[i, j])) for j in top])
    return out

Embeddings cover 100 docs; aligned with BM25


### Query run

In [17]:
qids = list(queries)
qtexts = list(queries.values())

def to_run(lists, name):
    return Run({q: dict(r) for q, r in zip(qids, lists)}, name=name)

runs = {'bm25': to_run(bm25_search(qtexts), 'bm25')}
print('bm25 done')

Split strings:   0%|          | 0/48 [00:00<?, ?it/s]

Reconstructing token strings:   0%|          | 0/48 [00:00<?, ?it/s]

bm25 done


In [18]:
# Dense runs: loads each query encoder once, then frees memory
import gc

for key in DENSE_MODELS:
    print(f'Retrieving with {key} ...')
    runs[key] = to_run(dense_search(key, qtexts), key)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print('Runs:', list(runs))

Retrieving with medcpt ...


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.49k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/226k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/706k [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Runs: ['bm25', 'medcpt', 'bge', 'qwen3']


In [19]:
# reciprocal rank fusion

for key in DENSE_MODELS:
    fused = fuse(runs=[runs['bm25'], runs[key]], method='rrf')
    fused.name = f'bm25+{key}'
    runs[fused.name] = fused

for name, run in runs.items():
    run.save(os.path.join(OUT_DIR, f'run_{name}.json'))

print('All runs:', list(runs))

/usr/local/lib/python3.13/dist-packages/ranx/normalization/min_max_norm.py:44: NumbaTypeSafetyWarning: unsafe cast from uint64 to int64. Precision may be lost.
  normalized_run[i] = _min_max_norm(run[q_ids[i]], invert)


All runs: ['bm25', 'medcpt', 'bge', 'qwen3', 'bm25+medcpt', 'bm25+bge', 'bm25+qwen3']


In [20]:
# Look at one question across runs
QID = 'q41'
print(queries[QID], '\n')
for name, run in runs.items():
    top = sorted(run.to_dict()[QID].items(), key=lambda x: -x[1])[:5]
    marks = ['*' if p in seeds[QID] else ('x' if p in traps[QID] else ' ') for p, _ in top]
    print(f'{name:12s}', '  '.join(f'{m}{p}' for m, (p, _) in zip(marks, top)))
print('\n* = seed (relevant), x = false-positive trap')

Which radiology studies gave the models the actual images, rather than only report text, to make a diagnosis? 

bm25          42630590   42684307   42745158   42750980   42645937
medcpt        42767241   42715525   42716901   42675279  *42649333
bge          x42733104  *42649333   42715525   42723869  *42566748
qwen3        *42773339  *42566748  *42649333   42793485   42766848
bm25+medcpt   42767241   42684307   42630590   42743860   42818963
bm25+bge      42743860   42645937  *42773339   42659896   42767241
bm25+qwen3   *42773339  *42566748   42818963   42743860   42582763

* = seed (relevant), x = false-positive trap


In [21]:
qrels_dict = {q: {p: 1 for p in s} for q, s in seeds.items() if s}
QRELS_SOURCE = 'seed PMIDs (provisional, incomplete)'

qrels = Qrels(qrels_dict)
answerable = list(qrels_dict)
print(f'Qrels: {QRELS_SOURCE} | {len(answerable)} answerable questions')

Qrels: seed PMIDs (provisional, incomplete) | 44 answerable questions


## Evaluation

In [22]:
def restrict(run, keep):
    d = run.to_dict()
    return Run({q: d[q] for q in keep if q in d}, name=run.name)

eval_runs = [restrict(r, answerable) for r in runs.values()]
report = compare(qrels, runs=eval_runs, metrics=METRICS, max_p=0.05)
print(f'Qrels: {QRELS_SOURCE}\n')
print(report)

with open(os.path.join(OUT_DIR, 'ranx_report.txt'), 'w') as f:
    f.write(f'Qrels: {QRELS_SOURCE}\n\n{report}\n')

Qrels: seed PMIDs (provisional, incomplete)

#    Model        Recall@5    Recall@10    NDCG@10    MRR@10    P@5
---  -----------  ----------  -----------  ---------  --------  --------
a    bm25         0.697       0.801        0.743ᵇ     0.804ᵇ    0.295ᵇ
b    medcpt       0.640       0.750        0.625      0.659     0.227
c    bge          0.725ᵇ      0.786        0.776ᵇ     0.868ᵇ    0.305ᵇ
d    qwen3        0.767ᵇ      0.843ᵇ       0.820ᵇ     0.914ᵇ    0.336ᵇᵉ
e    bm25+medcpt  0.715ᵇ      0.801        0.784ᵇ     0.889ᵇ    0.282ᵇ
f    bm25+bge     0.775ᵇᵉ     0.829ᵇ       0.806ᵃᵇ    0.893ᵃᵇ   0.332ᵇᵉ
g    bm25+qwen3   0.796ᵃᵇᶜᵉ   0.862ᵃᵇᶜᵉ    0.835ᵃᵇ    0.922ᵃᵇ   0.355ᵃᵇᵉ


In [23]:
# Per-query scores
rows = []
for r in eval_runs:
    sc = evaluate(qrels, r, ['recall@10', 'ndcg@10'], return_mean=False)
    for i, q in enumerate(qrels.keys()):
        rows.append({'run': r.name, 'qid': q, 'type': qtype[q],
                     'recall@10': float(sc['recall@10'][i]), 'ndcg@10': float(sc['ndcg@10'][i])})
per_query = pd.DataFrame(rows)
per_query.to_csv(os.path.join(OUT_DIR, 'metrics_per_query.csv'), index=False)

# nDCG@10 by question type
per_query.pivot_table(index='run', columns='type', values='ndcg@10').round(3)

type,hard_negative_contrast,multi_paper,single_paraphrased,single_specific
run,,,,
bge,0.337,0.620,0.850,1.000
bm25,0.444,0.623,0.658,1.000
bm25+bge,0.337,0.699,0.865,1.000
bm25+medcpt,0.465,0.600,0.863,1.000
bm25+qwen3,0.501,0.744,0.856,1.000
medcpt,0.354,0.444,0.635,0.871
qwen3,0.515,0.659,0.913,1.000


In [24]:
# Hardest questions: best nDCG@10 any run achieved (good starting point for error analysis)
best = per_query.groupby('qid')['ndcg@10'].max().sort_values().head(10)
pd.DataFrame({'best_ndcg@10': best.round(3), 'type': best.index.map(qtype),
              'question': best.index.map(queries)})

,best_ndcg@10,type,question
qid,,,
q41,0.469,hard_negative_contrast,Which radiology studies gave the models the ac...
q39,0.492,multi_paper,Which studies evaluated open-weight or locally...
q43,0.520,hard_negative_contrast,Which studies tested GPT-4o's ability to inter...
q36,0.690,multi_paper,Which papers propose new benchmarks or audit e...
q26,0.744,multi_paper,Which studies compared multimodal LLMs or VLMs...
q42,0.765,hard_negative_contrast,Which studies evaluated multimodal models on s...
q34,0.775,multi_paper,Which studies examined how prompting strategy ...
q38,0.788,multi_paper,Which studies evaluated AI models on spine ima...
q40,0.802,multi_paper,"Which papers are reviews (systematic, scoping,..."


In [25]:
def top_k(run, qid, k):
    return [p for p, _ in sorted(run.to_dict()[qid].items(), key=lambda x: -x[1])[:k]]

trap_qids = [q for q in qids if traps[q]]
trap_rows = []
for r in runs.values():
    hits = [any(p in traps[q] for p in top_k(r, q, 5)) for q in trap_qids]
    trap_rows.append({'run': r.name, 'trap_rate@5': round(float(np.mean(hits)), 3),
                      'questions_with_trap': f'{sum(hits)}/{len(hits)}'})
trap_df = pd.DataFrame(trap_rows)
trap_df.to_csv(os.path.join(OUT_DIR, 'trap_rate.csv'), index=False)
trap_df

,run,trap_rate@5,questions_with_trap
0,bm25,0.625,5/8
1,medcpt,0.250,2/8
2,bge,0.500,4/8
3,qwen3,0.375,3/8
4,bm25+medcpt,0.625,5/8
5,bm25+bge,0.500,4/8
6,bm25+qwen3,0.500,4/8
